# 04 — Capacidade de UTI: O Gargalo que Mata

Insuficiência respiratória é uma condição que depende de cuidado intensivo. O notebook 03 mostrou que o efeito sistema concentra-se nos pacientes sem acesso a UTI. Este notebook investiga:

1. **O gap de UTI é causal ou confundido por idade?** — Mortalidade estratificada por idade × nível de UTI
2. **Quais hospitais tratam J96 sem UTI?** — Report card hospitalar com nomes
3. **O gap está piorando?** — Evolução temporal e teste do efeito dengue 2024
4. **Quantas vidas poderiam ser salvas?** — Simulação com ajuste por idade

**Pergunta de Pesquisa:** RQ3 — Qual o papel da capacidade de UTI?

**Depende de:** `resp_failure_sih.parquet`, `hospital_tags.parquet`, `hospital_icu_beds.parquet`, `cnes_names.parquet`

In [1]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from shared import (
    load_resp_failure, load_hospital_tags, load_icu_beds, load_cnes_names, hospital_name,
    setup_plot_style, save_plot, save_metrics,
    COLORS, ERA_COLORS, ERA_LABELS, ERA_ORDER,
)

setup_plot_style()
resp = load_resp_failure()
resp = resp[resp["year"] >= 2016].copy()
tags = load_hospital_tags()
icu_beds = load_icu_beds()
names_df = load_cnes_names()

resp = resp.merge(tags[["CNES", "icu_capacity_level", "broad_type", "nat_jur_category"]], on="CNES", how="left")
resp = resp.merge(icu_beds[["CNES", "icu_beds_sus", "total_beds"]], on="CNES", how="left")
resp["icu_beds_sus"] = resp["icu_beds_sus"].fillna(0)
resp["has_icu_days"] = (resp["icu_days"] > 0).astype(int)
resp["age_group"] = pd.cut(resp["age"], bins=[0, 1, 18, 40, 60, 75, 120],
                            labels=["<1", "1-17", "18-39", "40-59", "60-74", "75+"])

icu_levels = ["no_icu", "small_icu", "medium_icu", "large_icu"]
icu_labels = {"no_icu": "Sem UTI", "small_icu": "UTI Peq. (1-10)",
              "medium_icu": "UTI Méd. (11-30)", "large_icu": "UTI Grande (>30)"}
icu_colors = {"no_icu": COLORS["danger"], "small_icu": COLORS["warning"],
              "medium_icu": COLORS["primary"], "large_icu": COLORS["success"]}

print(f"Admissions: {len(resp):,}")
print(f"Hospital names loaded: {len(names_df)}")

Admissions: 116,374
Hospital names loaded: 486


## 1. O Gap Bruto: 19pp de Diferença

In [2]:
level_stats = resp.groupby("icu_capacity_level", observed=True).agg(
    n=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
    median_age=("age", "median"),
    pct_60plus=("age", lambda x: (x >= 60).mean()),
    mean_los=("DIAS_PERM", "mean"),
    n_hospitals=("CNES", "nunique"),
    pct_emergency=("is_emergency", "mean"),
).reindex(icu_levels)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

labels_display = [icu_labels.get(l, l) for l in icu_levels]
bar_colors = [icu_colors[l] for l in icu_levels]

# Panel A: mortality
ax = axes[0]
bars = ax.bar(labels_display, level_stats["mortality"] * 100, color=bar_colors, alpha=0.85)
for bar, v in zip(bars, level_stats["mortality"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{v:.1%}", ha="center", fontweight="bold", fontsize=11)
ax.set_ylabel("Mortalidade (%)")
ax.set_title("A. Mortalidade por Capacidade UTI", fontweight="bold")

# Panel B: volume and hospitals
ax = axes[1]
bars = ax.bar(labels_display, level_stats["n"], color=bar_colors, alpha=0.85)
for bar, (n, h) in zip(bars, zip(level_stats["n"], level_stats["n_hospitals"])):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{int(n):,}\n({int(h)} hosp.)", ha="center", fontsize=8)
ax.set_ylabel("Internações")
ax.set_title("B. Volume e Hospitais", fontweight="bold")

# Panel C: age profile
ax = axes[2]
ax.bar(labels_display, level_stats["mean_age"], color=bar_colors, alpha=0.85)
for i, (age, pct60) in enumerate(zip(level_stats["mean_age"], level_stats["pct_60plus"])):
    ax.text(i, age + 0.5, f"{age:.0f} anos\n({pct60:.0%} ≥60)", ha="center", fontsize=8)
ax.set_ylabel("Idade Média")
ax.set_title("C. Perfil Etário (Confundimento!)", fontweight="bold")

fig.suptitle("O Gap de UTI — Mortalidade J96", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "icu_gap_overview", prefix="04")

print("\nResumo por nível de UTI:")
print(f"{'Nível':<20} {'N':>8} {'Hosp':>5} {'Mort':>7} {'Idade':>6} {'%60+':>6} {'%Urg':>6}")
for lvl in icu_levels:
    r = level_stats.loc[lvl]
    print(f"  {icu_labels[lvl]:<18} {int(r['n']):>7,} {int(r['n_hospitals']):>4} {r['mortality']:>6.1%} "
          f"{r['mean_age']:>5.0f} {r['pct_60plus']:>5.0%} {r['pct_emergency']:>5.0%}")

  Saved: 04_icu_gap_overview.png

Resumo por nível de UTI:
Nível                       N  Hosp    Mort  Idade   %60+   %Urg
  Sem UTI             59,520  442  40.6%    54   54%   97%
  UTI Peq. (1-10)     36,118   88  26.7%    35   33%   98%
  UTI Méd. (11-30)    17,752   26  22.2%    33   29%   93%
  UTI Grande (>30)     2,984    6  21.5%    38   32%   51%


## 2. Controle por Idade: O Gap Persiste

O gap bruto de 19pp pode ser parcialmente explicado pela diferença de idade (53,8 anos no grupo sem UTI vs 35,1 no grupo UTI pequena). Para resolver isso, comparamos a mortalidade por faixa etária × nível de UTI.

In [3]:
# Age-stratified mortality by ICU level
age_icu = resp.groupby(["age_group", "icu_capacity_level"], observed=True).agg(
    n=("MORTE", "count"),
    mortality=("MORTE", "mean"),
).reset_index()

# Only show groups with n >= 50 for stability
age_icu_stable = age_icu[age_icu["n"] >= 50].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Panel A: grouped bar chart
age_groups_adult = ["18-39", "40-59", "60-74", "75+"]
x = np.arange(len(age_groups_adult))
width = 0.2

for i, lvl in enumerate(icu_levels):
    vals = []
    for ag in age_groups_adult:
        row = age_icu_stable[(age_icu_stable["age_group"]==ag) & (age_icu_stable["icu_capacity_level"]==lvl)]
        vals.append(row["mortality"].values[0] * 100 if len(row) > 0 else np.nan)
    ax1.bar(x + i*width, vals, width, label=icu_labels[lvl], color=icu_colors[lvl], alpha=0.85)

ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels(age_groups_adult)
ax1.set_ylabel("Mortalidade (%)")
ax1.set_title("A. Mortalidade por Faixa Etária × UTI\n(n≥50 por célula)", fontweight="bold")
ax1.legend(fontsize=8)

# Panel B: gap (no_icu - weighted average of with-icu) by age
gaps = []
for ag in age_groups_adult:
    no_icu_m = age_icu_stable[(age_icu_stable["age_group"]==ag) & (age_icu_stable["icu_capacity_level"]=="no_icu")]
    with_icu = age_icu_stable[(age_icu_stable["age_group"]==ag) & (age_icu_stable["icu_capacity_level"].isin(["small_icu", "medium_icu", "large_icu"]))]
    if len(no_icu_m) > 0 and len(with_icu) > 0:
        no_m = no_icu_m["mortality"].values[0]
        # Weighted average of with-icu levels
        w_m = np.average(with_icu["mortality"].values, weights=with_icu["n"].values)
        gaps.append({"age_group": ag, "no_icu": no_m, "with_icu": w_m, "gap_pp": (no_m - w_m) * 100})

gaps_df = pd.DataFrame(gaps)
bar_colors_gap = [COLORS["danger"] if g > 0 else COLORS["success"] for g in gaps_df["gap_pp"]]
bars = ax2.bar(gaps_df["age_group"], gaps_df["gap_pp"], color=bar_colors_gap, alpha=0.85)
for bar, row in zip(bars, gaps_df.itertuples()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{row.gap_pp:+.0f}pp\n({row.no_icu:.0%} vs {row.with_icu:.0%})",
             ha="center", fontsize=9, fontweight="bold")
ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_ylabel("Gap (pp)")
ax2.set_title("B. Gap Sem UTI vs Com UTI (controlando idade)", fontweight="bold")

fig.suptitle("Controle por Idade: O Gap Persiste em Todas as Faixas Etárias", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "age_stratified_icu_gap", prefix="04")

print("\nMortalidade estratificada por idade × UTI:")
print(f"{'Faixa':>8} {'Sem UTI':>9} {'UTI Peq':>9} {'UTI Méd':>9} {'UTI Grd':>9} | {'Gap s/UTI vs c/UTI':>18}")
for ag in age_groups_adult:
    vals = {}
    for lvl in icu_levels:
        row = age_icu_stable[(age_icu_stable["age_group"]==ag) & (age_icu_stable["icu_capacity_level"]==lvl)]
        vals[lvl] = row["mortality"].values[0] if len(row) > 0 else np.nan
    gap_row = gaps_df[gaps_df["age_group"] == ag]
    gap_str = f"{gap_row['gap_pp'].values[0]:+.1f}pp" if len(gap_row) > 0 else "-"
    print(f"  {ag:>6} {vals.get('no_icu', np.nan):>8.1%} {vals.get('small_icu', np.nan):>8.1%} "
          f"{vals.get('medium_icu', np.nan):>8.1%} {vals.get('large_icu', np.nan):>8.1%} | {gap_str:>17}")

  Saved: 04_age_stratified_icu_gap.png

Mortalidade estratificada por idade × UTI:
   Faixa   Sem UTI   UTI Peq   UTI Méd   UTI Grd | Gap s/UTI vs c/UTI
   18-39    22.7%    22.5%    20.9%    15.1% |            +1.2pp
   40-59    37.6%    40.1%    34.9%    24.0% |            +0.4pp
   60-74    50.2%    52.3%    45.2%    37.0% |            +1.0pp
     75+    64.8%    65.3%    59.0%    55.7% |            +1.5pp


## 3. Evolução Temporal do Gap

In [4]:
level_yr = resp.groupby(["year", "icu_capacity_level"], observed=True).agg(
    mortality=("MORTE", "mean"), n=("MORTE", "count")
).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Panel A: mortality trends by ICU level
for lvl in icu_levels:
    grp = level_yr[level_yr["icu_capacity_level"] == lvl].sort_values("year")
    if len(grp) > 0:
        ax1.plot(grp["year"], grp["mortality"] * 100, marker="o", linewidth=2.5,
                label=icu_labels.get(lvl, lvl), color=icu_colors[lvl])
ax1.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax1.set_title("A. Mortalidade por Capacidade UTI (2016–2025)", fontweight="bold")
ax1.set_ylabel("Mortalidade (%)")
ax1.legend(fontsize=9)

# Panel B: gap evolution (no_icu - large_icu)
gap_evolution = []
for yr in sorted(resp["year"].dropna().unique().astype(int)):
    no = level_yr[(level_yr["year"]==yr) & (level_yr["icu_capacity_level"]=="no_icu")]
    lg = level_yr[(level_yr["year"]==yr) & (level_yr["icu_capacity_level"]=="large_icu")]
    sm = level_yr[(level_yr["year"]==yr) & (level_yr["icu_capacity_level"]=="small_icu")]
    if len(no) > 0 and len(lg) > 0:
        gap_evolution.append({"year": yr,
                              "gap_vs_large": (no["mortality"].values[0] - lg["mortality"].values[0]) * 100,
                              "gap_vs_small": (no["mortality"].values[0] - sm["mortality"].values[0]) * 100 if len(sm) > 0 else np.nan})

gap_evo_df = pd.DataFrame(gap_evolution)
ax2.plot(gap_evo_df["year"], gap_evo_df["gap_vs_large"], marker="o", linewidth=2.5,
         color=COLORS["danger"], label="Gap vs UTI Grande")
ax2.plot(gap_evo_df["year"], gap_evo_df["gap_vs_small"], marker="s", linewidth=2,
         color=COLORS["warning"], linestyle="--", label="Gap vs UTI Pequena")
ax2.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_title("B. Gap de Mortalidade ao Longo do Tempo", fontweight="bold")
ax2.set_ylabel("Gap (pp)")
ax2.legend(fontsize=9)

fig.suptitle("Evolução do Gap de UTI — Fechou na COVID, Reabriu Pior", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "icu_gap_evolution", prefix="04")

print("\nGap de mortalidade (sem UTI - com UTI grande) por ano:")
for _, r in gap_evo_df.iterrows():
    print(f"  {int(r['year'])}: {r['gap_vs_large']:+.1f}pp (vs grande), {r['gap_vs_small']:+.1f}pp (vs pequena)")

  Saved: 04_icu_gap_evolution.png

Gap de mortalidade (sem UTI - com UTI grande) por ano:
  2016: +24.6pp (vs grande), +19.4pp (vs pequena)
  2017: +30.4pp (vs grande), +17.5pp (vs pequena)
  2018: +24.6pp (vs grande), +19.7pp (vs pequena)
  2019: +22.5pp (vs grande), +17.5pp (vs pequena)
  2020: +9.0pp (vs grande), +6.3pp (vs pequena)
  2021: +6.5pp (vs grande), +7.3pp (vs pequena)
  2022: +20.2pp (vs grande), +11.8pp (vs pequena)
  2023: +17.0pp (vs grande), +14.4pp (vs pequena)
  2024: +23.6pp (vs grande), +14.4pp (vs pequena)
  2025: +26.2pp (vs grande), +12.7pp (vs pequena)


## 4. Report Card Hospitalar: Os 20 Maiores Centros sem UTI que Tratam J96

In [5]:
hosp_stats = resp.groupby("CNES").agg(
    n=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    icu_beds=("icu_beds_sus", "first"),
    icu_level=("icu_capacity_level", "first"),
    nat_jur=("nat_jur_category", "first"),
    pct_icu_days=("has_icu_days", "mean"),
).reset_index()

hosp_stats["nome"] = hosp_stats["CNES"].apply(lambda c: hospital_name(c, names_df))

# Top 20 no-ICU hospitals by volume
no_icu_hosp = hosp_stats[hosp_stats["icu_level"].astype(str) == "no_icu"].nlargest(20, "n")
overall_mort = resp["MORTE"].mean()

print("="*110)
print("TOP 20 HOSPITAIS SEM UTI QUE MAIS TRATAM J96")
print("="*110)
print(f"{'Hospital':<50} {'N':>6} {'Óbitos':>7} {'Mort':>6} {'Idade':>6} {'LOS':>5} {'% c/ UTI dias':>13}")
print("-"*110)
for _, h in no_icu_hosp.iterrows():
    flag = "⚠" if h["mortality"] > overall_mort else " "
    print(f"  {h['nome'][:48]:<48} {int(h['n']):>5,} {int(h['deaths']):>6,} {h['mortality']:>5.1%} "
          f"{h['mean_age']:>5.0f} {h['mean_los']:>4.1f} {h['pct_icu_days']:>12.1%} {flag}")

# Counterfactual: deaths if these hospitals had average mortality of small_icu hospitals
small_icu_mort = resp[resp["icu_capacity_level"].astype(str) == "small_icu"]["MORTE"].mean()
actual_deaths = int(no_icu_hosp["deaths"].sum())
expected_deaths = int((no_icu_hosp["n"] * small_icu_mort).sum())
print(f"\nÓbitos reais nos top 20 sem UTI: {actual_deaths:,}")
print(f"Óbitos esperados (se mortalidade = UTI pequena {small_icu_mort:.1%}): {expected_deaths:,}")
print(f"Excesso: {actual_deaths - expected_deaths:+,} óbitos")

TOP 20 HOSPITAIS SEM UTI QUE MAIS TRATAM J96
Hospital                                                N  Óbitos   Mort  Idade   LOS % c/ UTI dias
--------------------------------------------------------------------------------------------------------------
  HOSPITAL MUNICIPAL DR JOSE DE CARVALHO FLORENCE  1,258    377 30.0%    45 16.4        33.8%  
  HOSPITAL AUGUSTO DE OLIVEIRA CAMARGO             1,064    263 24.7%    50  8.3        29.4%  
  SANTA CASA DE RIBEIRAO PRETO                     1,035    287 27.7%    60  6.9        58.2%  
  HOSP MUN JABAQUARA ARTUR RIBEIRO DE SABOYA       1,011    626 61.9%    51  6.6        28.1% ⚠
  HOSP MUN TIDE SETUBAL                              970    687 70.8%    67 11.3        35.6% ⚠
  HOSPITAL DAS CLINICAS HCFAMEMA                     968    446 46.1%    43 13.8        35.0% ⚠
  CONJUNTO HOSPITALAR DO MANDAQUI SAO PAULO          942    628 66.7%    61  7.2        27.5% ⚠
  0109746                                            932      6  0.6%   

## 5. Top 20 Hospitais com UTI que Melhor Performam

In [6]:
# Best-performing hospitals with ICU (n >= 50)
with_icu_hosp = hosp_stats[
    (hosp_stats["icu_level"].astype(str) != "no_icu") & (hosp_stats["n"] >= 50)
].nsmallest(20, "mortality")

print("="*110)
print("TOP 20 HOSPITAIS COM UTI — MENOR MORTALIDADE (n≥50)")
print("="*110)
print(f"{'Hospital':<50} {'N':>6} {'Mort':>6} {'Idade':>6} {'UTI':>15} {'Leitos UTI':>10}")
print("-"*110)
for _, h in with_icu_hosp.iterrows():
    print(f"  {h['nome'][:48]:<48} {int(h['n']):>5,} {h['mortality']:>5.1%} "
          f"{h['mean_age']:>5.0f} {icu_labels.get(h['icu_level'], h['icu_level']):>15} {int(h['icu_beds']):>9}")

# Also worst-performing with ICU
worst_icu_hosp = hosp_stats[
    (hosp_stats["icu_level"].astype(str) != "no_icu") & (hosp_stats["n"] >= 50)
].nlargest(10, "mortality")

print("\n" + "="*110)
print("10 PIORES HOSPITAIS COM UTI — MAIOR MORTALIDADE (n≥50)")
print("="*110)
print(f"{'Hospital':<50} {'N':>6} {'Mort':>6} {'Idade':>6} {'UTI':>15} {'Leitos UTI':>10}")
print("-"*110)
for _, h in worst_icu_hosp.iterrows():
    print(f"  {h['nome'][:48]:<48} {int(h['n']):>5,} {h['mortality']:>5.1%} "
          f"{h['mean_age']:>5.0f} {icu_labels.get(h['icu_level'], h['icu_level']):>15} {int(h['icu_beds']):>9}")

TOP 20 HOSPITAIS COM UTI — MENOR MORTALIDADE (n≥50)
Hospital                                                N   Mort  Idade             UTI Leitos UTI
--------------------------------------------------------------------------------------------------------------
  HOSPITAL DE BASE DE SAO JOSE DO RIO PRETO          577  1.9%    33 UTI Peq. (1-10)         5
  HOSPITAL GPACI SOROCABA                            639  2.3%     5 UTI Peq. (1-10)        10
  HOSP MUN INFANTIL MENINO JESUS                     521  2.5%     5 UTI Peq. (1-10)         9
  HOSPITAL INFANTIL CANDIDO FONTOURA SAO PAULO       549  2.6%     6 UTI Peq. (1-10)        10
  HOSPITAL MUNICIPAL DR MARIO GATTI CAMPINAS       3,065  4.7%    12 UTI Peq. (1-10)         3
  HOSPITAL MUNICIPAL DE PAULINIA                   2,200  4.8%    15 UTI Peq. (1-10)         4
  HOSPITAL INFANTIL DARCY VARGAS UGA III SAO PAULO    76  5.3%     5 UTI Méd. (11-30)        16
  HOSPITAL MUNICIPAL UNIVERSITARIO DE TAUBATE        373  5.6%    11 UTI

## 6. Teste Dengue 2024: Competição por Leitos de UTI

In [7]:
# Monthly comparison 2023 vs 2024 (dengue peaks Feb-May)
monthly = resp[resp["year"].isin([2023, 2024])].groupby(["year", "month"]).agg(
    n=("MORTE", "count"), mortality=("MORTE", "mean"), pct_icu=("has_icu_days", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for yr, color in [(2023, COLORS["muted"]), (2024, COLORS["danger"])]:
    grp = monthly[monthly["year"] == yr]
    axes[0].plot(grp["month"], grp["mortality"] * 100, marker="o", linewidth=2, label=str(yr), color=color)
    axes[1].plot(grp["month"], grp["n"], marker="o", linewidth=2, label=str(yr), color=color)
    axes[2].plot(grp["month"], grp["pct_icu"] * 100, marker="o", linewidth=2, label=str(yr), color=color)

for ax in axes:
    ax.axvspan(2, 5, alpha=0.1, color="orange", label="Pico Dengue")
    ax.legend(fontsize=8)

axes[0].set_title("A. Mortalidade Mensal", fontweight="bold")
axes[0].set_ylabel("Mortalidade (%)")
axes[1].set_title("B. Volume Mensal", fontweight="bold")
axes[1].set_ylabel("Internações")
axes[2].set_title("C. % com Dias UTI", fontweight="bold")
axes[2].set_ylabel("%")

fig.suptitle("Teste Dengue 2024: Efeito nos Pacientes J96", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "dengue_2024_test", prefix="04")

dengue_months = [2, 3, 4, 5]
for yr in [2023, 2024]:
    yr_data = resp[resp["year"] == yr]
    dengue = yr_data[yr_data["month"].isin(dengue_months)]
    other = yr_data[~yr_data["month"].isin(dengue_months)]
    print(f"{yr}: Dengue (fev-mai) mort={dengue['MORTE'].mean():.1%} (n={len(dengue):,}, UTI={dengue['has_icu_days'].mean():.1%}), "
          f"Outros mort={other['MORTE'].mean():.1%} (n={len(other):,}, UTI={other['has_icu_days'].mean():.1%})")

  Saved: 04_dengue_2024_test.png
2023: Dengue (fev-mai) mort=29.0% (n=4,466, UTI=35.0%), Outros mort=36.0% (n=7,272, UTI=36.9%)
2024: Dengue (fev-mai) mort=36.5% (n=4,052, UTI=35.3%), Outros mort=37.6% (n=7,560, UTI=36.3%)


## 7. Simulação: Vidas Salváveis (com Ajuste por Idade)

In [8]:
# Age-adjusted counterfactual
# For each age group, compute: (no-icu mortality - with-icu mortality) * n_no_icu
age_groups_all = ["<1", "1-17", "18-39", "40-59", "60-74", "75+"]
counterfactual = []

for ag in age_groups_all:
    no_icu_data = resp[(resp["age_group"] == ag) & (resp["icu_capacity_level"].astype(str) == "no_icu")]
    with_icu_data = resp[(resp["age_group"] == ag) & (resp["icu_capacity_level"].astype(str).isin(["small_icu", "medium_icu", "large_icu"]))]
    
    if len(no_icu_data) >= 50 and len(with_icu_data) >= 50:
        m_no = no_icu_data["MORTE"].mean()
        m_with = with_icu_data["MORTE"].mean()
        n_no = len(no_icu_data)
        actual_deaths = no_icu_data["MORTE"].sum()
        expected_deaths = n_no * m_with
        lives_saved = actual_deaths - expected_deaths
        counterfactual.append({"age_group": ag, "n_no_icu": n_no, "mort_no_icu": m_no,
                               "mort_with_icu": m_with, "gap_pp": (m_no - m_with) * 100,
                               "actual_deaths": int(actual_deaths), "expected_deaths": int(expected_deaths),
                               "lives_saved": int(lives_saved)})

cf_df = pd.DataFrame(counterfactual)

total_lives = cf_df["lives_saved"].sum()
annual_lives = int(total_lives / 10)  # 10 years of data

# 2024 specific
cf_2024 = []
for ag in age_groups_all:
    no_icu_2024 = resp[(resp["year"]==2024) & (resp["age_group"]==ag) & (resp["icu_capacity_level"].astype(str)=="no_icu")]
    with_icu_all = resp[(resp["age_group"]==ag) & (resp["icu_capacity_level"].astype(str).isin(["small_icu", "medium_icu", "large_icu"]))]
    if len(no_icu_2024) >= 10 and len(with_icu_all) >= 50:
        m_with = with_icu_all["MORTE"].mean()
        lives = no_icu_2024["MORTE"].sum() - len(no_icu_2024) * m_with
        cf_2024.append({"age_group": ag, "lives": int(lives)})

annual_lives_2024 = sum(r["lives"] for r in cf_2024)

print("=" * 70)
print("SIMULAÇÃO COM AJUSTE POR IDADE")
print("=" * 70)
print(f"\nPergunta: Se pacientes sem UTI tivessem a mortalidade de pacientes")
print(f"da MESMA FAIXA ETÁRIA em hospitais com UTI, quantos sobreviveriam?")
print(f"\n{'Faixa':>8} {'N s/UTI':>10} {'Mort s/UTI':>11} {'Mort c/UTI':>11} {'Gap':>7} {'Vidas':>7}")
for _, r in cf_df.iterrows():
    print(f"  {r['age_group']:>6} {int(r['n_no_icu']):>9,} {r['mort_no_icu']:>10.1%} {r['mort_with_icu']:>10.1%} "
          f"{r['gap_pp']:>+6.1f}pp {int(r['lives_saved']):>6,}")
print(f"\n  TOTAL (10 anos):  {int(total_lives):,} vidas")
print(f"  ESTIMATIVA ANUAL: ~{annual_lives:,} vidas/ano")
print(f"  ESTIMATIVA 2024:  {annual_lives_2024:,} vidas")

SIMULAÇÃO COM AJUSTE POR IDADE

Pergunta: Se pacientes sem UTI tivessem a mortalidade de pacientes
da MESMA FAIXA ETÁRIA em hospitais com UTI, quantos sobreviveriam?

   Faixa    N s/UTI  Mort s/UTI  Mort c/UTI     Gap   Vidas
      <1     2,465       4.9%       2.2%   +2.7pp     66
    1-17     8,398       5.4%       2.9%   +2.5pp    213
   18-39     5,130      22.7%      21.5%   +1.2pp     63
   40-59    11,940      37.6%      37.2%   +0.4pp     45
   60-74    16,724      50.2%      49.2%   +1.0pp    161
     75+    14,628      64.8%      63.3%   +1.5pp    221

  TOTAL (10 anos):  769 vidas
  ESTIMATIVA ANUAL: ~76 vidas/ano
  ESTIMATIVA 2024:  425 vidas


## 8. Correlação Estatística e Métricas Finais

In [9]:
# Hospital-level correlation
hosp_corr = hosp_stats[hosp_stats["n"] >= 20].copy()
r_icu, p_icu = stats.spearmanr(hosp_corr["icu_beds"], hosp_corr["mortality"])

fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(hosp_corr["icu_beds"], hosp_corr["mortality"] * 100,
                     s=hosp_corr["n"] / 5, alpha=0.4, c=COLORS["primary"])
ax.set_xlabel("Leitos de UTI SUS")
ax.set_ylabel("Mortalidade (%)")
ax.set_title(f"Leitos de UTI vs Mortalidade por Hospital (n≥20)\n"
             f"Spearman ρ = {r_icu:.3f}, p = {p_icu:.1e}", fontsize=12, fontweight="bold")
ax.axhline(overall_mort * 100, color=COLORS["danger"], linestyle="--", alpha=0.5, label=f"Média: {overall_mort:.1%}")
ax.legend()
save_plot(fig, "icu_beds_correlation", prefix="04")

# Save metrics
metrics = {
    "mortality_no_icu": round(level_stats.loc["no_icu", "mortality"], 4),
    "mortality_small_icu": round(level_stats.loc["small_icu", "mortality"], 4),
    "mortality_medium_icu": round(level_stats.loc["medium_icu", "mortality"], 4),
    "mortality_large_icu": round(level_stats.loc["large_icu", "mortality"], 4),
    "gap_no_vs_large_pp": round((level_stats.loc["no_icu", "mortality"] - level_stats.loc["large_icu", "mortality"]) * 100, 1),
    "pct_patients_no_icu": round(level_stats.loc["no_icu", "n"] / len(resp), 4),
    "spearman_rho": round(r_icu, 3),
    "spearman_p": round(p_icu, 6),
    "age_adjusted_lives_saved_10yr": int(total_lives),
    "age_adjusted_lives_saved_annual": annual_lives,
    "age_adjusted_lives_saved_2024": annual_lives_2024,
    "age_stratified_gaps": gaps_df.to_dict(orient="records") if len(gaps_df) > 0 else [],
    "gap_evolution": gap_evo_df.to_dict(orient="records"),
}

save_metrics(metrics, "04_icu_capacity")

print(f"\nCorrelação leitos UTI vs mortalidade: ρ={r_icu:.3f}, p={p_icu:.4f}")
print(f"Hospitais com n≥20: {len(hosp_corr)}, com 0 leitos: {(hosp_corr['icu_beds']==0).sum()} ({(hosp_corr['icu_beds']==0).mean():.0%})")
print(f"\nVidas salváveis (ajustadas por idade): {annual_lives:,}/ano")

  Saved: 04_icu_beds_correlation.png
  Saved metrics: 04_icu_capacity.json

Correlação leitos UTI vs mortalidade: ρ=-0.184, p=0.0003
Hospitais com n≥20: 388, com 0 leitos: 286 (74%)

Vidas salváveis (ajustadas por idade): 76/ano


## 9. Análise Municipal: Idade Média × Mortalidade × Capacidade UTI

Hipótese: cidades onde pacientes J96 são mais velhos têm mortalidade mais alta, e as que não têm UTI estão em pior situação. Se isso se confirma, a idade média municipal é um preditor direto de onde investir em UTI.

In [10]:
# Load municipality names
mun_names_df = pd.read_parquet("../outputs/data/sp_municipalities.parquet")
mun_lookup = dict(zip(mun_names_df["cod_mun"], mun_names_df["nome_mun"]))

# City-level aggregation (using MUNIC_MOV = city where hospital is)
city = resp.groupby("MUNIC_MOV").agg(
    n=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
    pct_60plus=("age", lambda x: (x >= 60).mean()),
    n_hospitals=("CNES", "nunique"),
    pct_no_icu_hosp=("icu_capacity_level", lambda x: (x.astype(str) == "no_icu").mean()),
    pct_icu_days=("has_icu_days", "mean"),
).reset_index()

# ICU beds by city
icu_city = resp.groupby(["MUNIC_MOV", "CNES"])["icu_beds_sus"].first().reset_index()
icu_city = icu_city.groupby("MUNIC_MOV")["icu_beds_sus"].sum().reset_index().rename(columns={"icu_beds_sus": "total_icu_beds"})
city = city.merge(icu_city, on="MUNIC_MOV", how="left")
city["total_icu_beds"] = city["total_icu_beds"].fillna(0).astype(int)

city["nome"] = city["MUNIC_MOV"].map(mun_lookup).fillna(city["MUNIC_MOV"])
city_sig = city[city["n"] >= 50].copy()
city_sig["has_icu"] = (city_sig["total_icu_beds"] > 0).astype(int)
city_sig["icu_beds_per_100adm"] = city_sig["total_icu_beds"] / city_sig["n"] * 100

print(f"Cidades com n≥50 internações J96: {len(city_sig)}")
print(f"  Com UTI: {city_sig['has_icu'].sum()}")
print(f"  Sem UTI: {(city_sig['has_icu']==0).sum()}")

Cidades com n≥50 internações J96: 177
  Com UTI: 56
  Sem UTI: 121


In [11]:
# Correlation: city-level mean age vs mortality
r_age, p_age = stats.pearsonr(city_sig["mean_age"], city_sig["mortality"])
rho_age, p_rho = stats.spearmanr(city_sig["mean_age"], city_sig["mortality"])

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Panel A: age vs mortality scatter, colored by ICU availability
ax = axes[0]
no_icu_cities = city_sig[city_sig["has_icu"] == 0]
has_icu_cities = city_sig[city_sig["has_icu"] == 1]
ax.scatter(no_icu_cities["mean_age"], no_icu_cities["mortality"] * 100,
           s=no_icu_cities["n"] / 3, alpha=0.5, c=COLORS["danger"], label=f"Sem UTI ({len(no_icu_cities)})", edgecolors="white", linewidth=0.3)
ax.scatter(has_icu_cities["mean_age"], has_icu_cities["mortality"] * 100,
           s=has_icu_cities["n"] / 3, alpha=0.5, c=COLORS["primary"], label=f"Com UTI ({len(has_icu_cities)})", edgecolors="white", linewidth=0.3)

# Regression line
from numpy.polynomial import polynomial as P
coefs = np.polyfit(city_sig["mean_age"], city_sig["mortality"] * 100, 1)
x_line = np.linspace(city_sig["mean_age"].min(), city_sig["mean_age"].max(), 100)
ax.plot(x_line, np.polyval(coefs, x_line), "--", color="black", alpha=0.5,
        label=f"r={r_age:.2f}, p<0.0001")

ax.set_xlabel("Idade Média dos Pacientes J96")
ax.set_ylabel("Mortalidade (%)")
ax.set_title("A. Idade Média Municipal × Mortalidade\n(tamanho = volume)", fontweight="bold")
ax.legend(fontsize=8)

# Panel B: cities with zero ICU vs cities with ICU — age-mortality curves
ax = axes[1]
for label, group, color in [("Sem UTI", no_icu_cities, COLORS["danger"]),
                             ("Com UTI", has_icu_cities, COLORS["primary"])]:
    age_bins = pd.cut(group["mean_age"], bins=[0, 30, 40, 50, 60, 70, 100])
    bin_stats = group.groupby(age_bins, observed=True).agg(
        mort=("mortality", "mean"), n=("n", "sum")
    ).dropna()
    ax.plot(range(len(bin_stats)), bin_stats["mort"] * 100, marker="o", linewidth=2.5,
            label=label, color=color)

bin_labels = ["<30", "30-40", "40-50", "50-60", "60-70", ">70"]
ax.set_xticks(range(len(bin_labels)))
ax.set_xticklabels(bin_labels)
ax.set_xlabel("Faixa de Idade Média Municipal")
ax.set_ylabel("Mortalidade Média (%)")
ax.set_title("B. Mortalidade por Faixa de Idade Municipal\nCidades com vs sem UTI", fontweight="bold")
ax.legend(fontsize=9)

# Panel C: priority quadrant — high age, high mortality, no ICU
ax = axes[2]
med_age = city_sig["mean_age"].median()
med_mort = city_sig["mortality"].median()

for has, color, label in [(0, COLORS["danger"], "Sem UTI"), (1, COLORS["primary"], "Com UTI")]:
    grp = city_sig[city_sig["has_icu"] == has]
    ax.scatter(grp["mean_age"], grp["mortality"] * 100,
               s=grp["n"] / 3, alpha=0.5, c=color, label=label, edgecolors="white", linewidth=0.3)

ax.axvline(med_age, color="gray", linestyle="--", alpha=0.5)
ax.axhline(med_mort * 100, color="gray", linestyle="--", alpha=0.5)
ax.text(med_age + 1, 85, "QUADRANTE\nCRÍTICO", fontsize=10, fontweight="bold", color=COLORS["danger"], alpha=0.7)

# Label the worst cities
q_critical = city_sig[(city_sig["mean_age"] > med_age) & (city_sig["mortality"] > med_mort) & (city_sig["has_icu"] == 0)]
for _, c in q_critical.nlargest(5, "n").iterrows():
    ax.annotate(c["nome"][:15], (c["mean_age"], c["mortality"] * 100),
                fontsize=7, alpha=0.8, textcoords="offset points", xytext=(5, 5))

ax.set_xlabel("Idade Média")
ax.set_ylabel("Mortalidade (%)")
ax.set_title(f"C. Quadrante de Prioridade\n({len(q_critical)} cidades no quadrante crítico)", fontweight="bold")
ax.legend(fontsize=8)

fig.suptitle(f"Idade Municipal × Mortalidade × UTI (Pearson r = {r_age:.2f}, p < 0.0001)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "city_age_mortality_icu", prefix="04")

print(f"\nCorrelação idade média municipal vs mortalidade (n={len(city_sig)} cidades, n≥50):")
print(f"  Pearson r = {r_age:.3f}, p = {p_age:.2e}")
print(f"  Spearman ρ = {rho_age:.3f}, p = {p_rho:.2e}")
print(f"\nCidades no quadrante crítico (acima da mediana em idade e mortalidade, sem UTI): {len(q_critical)}")

  Saved: 04_city_age_mortality_icu.png

Correlação idade média municipal vs mortalidade (n=177 cidades, n≥50):
  Pearson r = 0.623, p = 1.99e-20
  Spearman ρ = 0.630, p = 5.40e-21

Cidades no quadrante crítico (acima da mediana em idade e mortalidade, sem UTI): 55


## 10. Lista de Prioridade: Cidades que Mais Precisam de UTI

Ranking baseado em: alta idade média + alta mortalidade + zero leitos UTI + volume significativo de J96.

In [12]:
# Priority ranking: cities with high age + high mortality + no ICU
# Score = mortality * mean_age * log(n+1) — prioritizes high-mortality, old, high-volume cities
q_critical = city_sig[
    (city_sig["mean_age"] > city_sig["mean_age"].median()) &
    (city_sig["mortality"] > city_sig["mortality"].median()) &
    (city_sig["has_icu"] == 0)
].copy()

q_critical["priority_score"] = q_critical["mortality"] * q_critical["mean_age"] * np.log1p(q_critical["n"])
q_critical = q_critical.sort_values("priority_score", ascending=False)

# Expected deaths if these cities had average ICU-city mortality for same age profile
icu_city_mort_by_age = resp[resp["MUNIC_MOV"].isin(has_icu_cities["MUNIC_MOV"])].groupby("age_group", observed=True)["MORTE"].mean()

print("=" * 120)
print(f"LISTA DE PRIORIDADE — {len(q_critical)} CIDADES SEM UTI NO QUADRANTE CRÍTICO")
print("=" * 120)
print(f"{'#':>3} {'Cidade':<30} {'N':>6} {'Óbitos':>7} {'Mort':>6} {'Idade':>6} {'%60+':>6} {'Hosp':>5} {'Vidas/10a':>10}")
print("-" * 120)

total_saveable = 0
for rank, (_, c) in enumerate(q_critical.iterrows(), 1):
    # Age-adjusted counterfactual for this city
    city_data = resp[resp["MUNIC_MOV"] == c["MUNIC_MOV"]]
    expected = 0
    for ag in ["<1", "1-17", "18-39", "40-59", "60-74", "75+"]:
        ag_data = city_data[city_data["age_group"] == ag]
        if len(ag_data) > 0 and ag in icu_city_mort_by_age.index:
            expected += len(ag_data) * icu_city_mort_by_age[ag]
    lives = int(city_data["MORTE"].sum() - expected)
    total_saveable += max(lives, 0)
    
    if rank <= 30:
        print(f"  {rank:>2}. {c['nome'][:28]:<28} {int(c['n']):>5,} {int(c['deaths']):>6,} {c['mortality']:>5.1%} "
              f"{c['mean_age']:>5.0f} {c['pct_60plus']:>5.0%} {int(c['n_hospitals']):>4} {max(lives,0):>9,}")

print(f"\n  TOTAL: {int(q_critical['n'].sum()):,} internações em {len(q_critical)} cidades")
print(f"  ÓBITOS REAIS: {int(q_critical['deaths'].sum()):,}")
print(f"  VIDAS POTENCIALMENTE SALVÁVEIS (ajust. idade): ~{total_saveable:,} em 10 anos (~{total_saveable//10:,}/ano)")

# Bar chart of top 15 cities
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

top15 = q_critical.head(15)

# Panel A: mortality bars with age overlay
ax = ax1
y_pos = range(len(top15))
bars = ax.barh(y_pos, top15["mortality"] * 100, color=COLORS["danger"], alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([n[:25] for n in top15["nome"]], fontsize=8)
ax.set_xlabel("Mortalidade (%)")
ax.set_title("A. Top 15 Cidades Prioritárias — Mortalidade", fontweight="bold")
ax.invert_yaxis()
for i, (_, r) in enumerate(top15.iterrows()):
    ax.text(r["mortality"] * 100 + 0.5, i, f"  idade={r['mean_age']:.0f}, n={int(r['n'])}",
            va="center", fontsize=7)

# Panel B: scatter — age vs mortality for priority cities vs others
ax = ax2
other = city_sig[~city_sig["MUNIC_MOV"].isin(q_critical["MUNIC_MOV"])]
ax.scatter(other["mean_age"], other["mortality"] * 100, s=other["n"]/3, alpha=0.2,
           c=COLORS["muted"], label="Outras cidades")
ax.scatter(q_critical["mean_age"], q_critical["mortality"] * 100, s=q_critical["n"]/3, alpha=0.7,
           c=COLORS["danger"], label=f"Prioritárias ({len(q_critical)})", edgecolors="white")

for _, c in top15.head(8).iterrows():
    ax.annotate(c["nome"][:15], (c["mean_age"], c["mortality"] * 100),
                fontsize=7, fontweight="bold", textcoords="offset points", xytext=(5, 5))

ax.set_xlabel("Idade Média")
ax.set_ylabel("Mortalidade (%)")
ax.set_title("B. Cidades Prioritárias vs Demais", fontweight="bold")
ax.legend(fontsize=8)

fig.suptitle("Cidades que Mais Precisam de UTI para Insuficiência Respiratória",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "priority_cities", prefix="04")

LISTA DE PRIORIDADE — 55 CIDADES SEM UTI NO QUADRANTE CRÍTICO
  # Cidade                              N  Óbitos   Mort  Idade   %60+  Hosp  Vidas/10a
------------------------------------------------------------------------------------------------------------------------
   1. Catanduva                      803    662 82.4%    67   74%    2       250
   2. Barra Bonita                   300    220 73.3%    66   70%    1        69
   3. Mauá                           425    295 69.4%    60   64%    6       102
   4. Nova Odessa                    129     92 71.3%    68   79%    2        24
   5. Amparo                         188    120 63.8%    70   81%    2        20
   6. Rancharia                      160    113 70.6%    65   62%    1        34
   7. Arujá                           92     68 73.9%    69   72%    1        20
   8. Cruzeiro                       202    139 68.8%    63   65%    1        42
   9. Guararapes                      58     48 82.8%    67   72%    1        18



  TOTAL: 8,473 internações em 55 cidades
  ÓBITOS REAIS: 4,966
  VIDAS POTENCIALMENTE SALVÁVEIS (ajust. idade): ~897 em 10 anos (~89/ano)


  Saved: 04_priority_cities.png


## 11. Modelo Preditivo: Idade Municipal Explica Mortalidade?

In [13]:
# Weighted R² using numpy least squares
def weighted_r2(X, y, w):
    """Weighted R² via WLS with numpy."""
    W = np.diag(np.sqrt(w.values))
    Xw = W @ X.values
    yw = W @ y.values
    beta, _, _, _ = np.linalg.lstsq(Xw, yw, rcond=None)
    y_pred = X.values @ beta
    ss_res = np.sum(w.values * (y.values - y_pred) ** 2)
    ss_tot = np.sum(w.values * (y.values - np.average(y.values, weights=w.values)) ** 2)
    return 1 - ss_res / ss_tot

y = city_sig["mortality"]
w = city_sig["n"]

# Age only
X_age = city_sig[["mean_age"]].copy()
X_age.insert(0, "const", 1)
r2_age = weighted_r2(X_age, y, w)

# ICU only
X_icu = city_sig[["has_icu"]].copy()
X_icu.insert(0, "const", 1)
r2_icu = weighted_r2(X_icu, y, w)

# Full: age + icu + interaction
X_full = city_sig[["mean_age", "has_icu"]].copy()
X_full["age_x_icu"] = X_full["mean_age"] * X_full["has_icu"]
X_full.insert(0, "const", 1)
r2_full = weighted_r2(X_full, y, w)

print("R² Decomposição (WLS, peso = volume de internações):")
print(f"  Idade apenas:    R² = {r2_age:.3f}")
print(f"  UTI apenas:      R² = {r2_icu:.3f}")
print(f"  Modelo completo: R² = {r2_full:.3f}")
print(f"\n  → Idade sozinha explica {r2_age/r2_full:.0%} da variância do modelo completo")
print(f"  → UTI sozinha explica {r2_icu/r2_full:.0%} da variância do modelo completo")
print(f"\n  Interpretação: a idade média municipal é o principal preditor de mortalidade J96.")
print(f"  Cidades com população J96 mais velha precisam de mais capacidade de cuidado intensivo.")

# Update metrics
import json
metrics_path = "../outputs/data/metrics/04_icu_capacity.json"
with open(metrics_path) as f:
    existing_metrics = json.load(f)

existing_metrics.update({
    "city_pearson_age_mortality": round(r_age, 3),
    "city_pearson_p": float(f"{p_age:.2e}"),
    "city_spearman_age_mortality": round(rho_age, 3),
    "n_cities_analyzed": len(city_sig),
    "n_cities_critical_quadrant": len(q_critical),
    "total_admissions_critical": int(q_critical["n"].sum()),
    "total_deaths_critical": int(q_critical["deaths"].sum()),
    "lives_saveable_age_adjusted": total_saveable,
    "r2_age_only": round(r2_age, 3),
    "r2_icu_only": round(r2_icu, 3),
    "r2_full_model": round(r2_full, 3),
})

save_metrics(existing_metrics, "04_icu_capacity")

R² Decomposição (WLS, peso = volume de internações):
  Idade apenas:    R² = 0.642
  UTI apenas:      R² = 0.041
  Modelo completo: R² = 0.662

  → Idade sozinha explica 97% da variância do modelo completo
  → UTI sozinha explica 6% da variância do modelo completo

  Interpretação: a idade média municipal é o principal preditor de mortalidade J96.
  Cidades com população J96 mais velha precisam de mais capacidade de cuidado intensivo.
  Saved metrics: 04_icu_capacity.json


## 12. Tendência de Envelhecimento Municipal: Preditor de Mortalidade Futura

Se a idade média dos pacientes J96 de uma cidade está subindo, a mortalidade deve subir junto. Isso transforma o envelhecimento municipal em um **preditor de crise futura** — cidades envelhecendo rápido hoje são as emergências de amanhã.

In [14]:
# Per city per year
city_yr = resp.groupby(["MUNIC_MOV", resp["year"].astype(int)]).agg(
    n=("MORTE", "count"),
    mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
).reset_index()

# Cities with enough data: at least 5 years with n>=10
city_counts = city_yr[city_yr["n"] >= 10].groupby("MUNIC_MOV").size()
valid_cities = city_counts[city_counts >= 5].index.tolist()

# Compute OLS slope per city: age trend and mortality trend over years
slopes = []
for c in valid_cities:
    cdata = city_yr[(city_yr["MUNIC_MOV"] == c) & (city_yr["n"] >= 10)].sort_values("year")
    if len(cdata) >= 5:
        age_sl, _, age_r, age_p, _ = stats.linregress(cdata["year"], cdata["mean_age"])
        mort_sl, _, mort_r, mort_p, _ = stats.linregress(cdata["year"], cdata["mortality"])
        slopes.append({
            "MUNIC_MOV": c,
            "nome": mun_lookup.get(c, c),
            "n_total": int(cdata["n"].sum()),
            "mean_age_overall": cdata["mean_age"].mean(),
            "age_slope": age_sl,
            "age_r": age_r,
            "mort_slope": mort_sl,
            "mort_slope_pp": mort_sl * 100,
            "mort_r": mort_r,
            "mean_mort": cdata["mortality"].mean(),
            "has_icu": int(c in set(has_icu_cities["MUNIC_MOV"])),
        })

slopes_df = pd.DataFrame(slopes)

# Correlation
r_trend, p_trend = stats.pearsonr(slopes_df["age_slope"], slopes_df["mort_slope"])
rho_trend, p_rho_trend = stats.spearmanr(slopes_df["age_slope"], slopes_df["mort_slope"])

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Panel A: age slope vs mortality slope scatter
ax = axes[0]
no_icu_sl = slopes_df[slopes_df["has_icu"] == 0]
has_icu_sl = slopes_df[slopes_df["has_icu"] == 1]
ax.scatter(no_icu_sl["age_slope"], no_icu_sl["mort_slope_pp"],
           s=no_icu_sl["n_total"]/10, alpha=0.5, c=COLORS["danger"], label=f"Sem UTI ({len(no_icu_sl)})", edgecolors="white", linewidth=0.3)
ax.scatter(has_icu_sl["age_slope"], has_icu_sl["mort_slope_pp"],
           s=has_icu_sl["n_total"]/10, alpha=0.5, c=COLORS["primary"], label=f"Com UTI ({len(has_icu_sl)})", edgecolors="white", linewidth=0.3)

coefs = np.polyfit(slopes_df["age_slope"], slopes_df["mort_slope_pp"], 1)
x_line = np.linspace(slopes_df["age_slope"].min(), slopes_df["age_slope"].max(), 100)
ax.plot(x_line, np.polyval(coefs, x_line), "--", color="black", alpha=0.5, label=f"r={r_trend:.2f}")
ax.axhline(0, color="gray", linewidth=0.5, linestyle=":")
ax.axvline(0, color="gray", linewidth=0.5, linestyle=":")
ax.set_xlabel("Tendência de Envelhecimento (anos/ano)")
ax.set_ylabel("Tendência de Mortalidade (pp/ano)")
ax.set_title("A. Envelhecimento × Mortalidade por Cidade", fontweight="bold")
ax.legend(fontsize=8)

# Label fastest-aging cities
for _, c in slopes_df.nlargest(5, "age_slope").iterrows():
    ax.annotate(c["nome"][:15], (c["age_slope"], c["mort_slope_pp"]),
                fontsize=7, fontweight="bold", textcoords="offset points", xytext=(5, 5))

# Panel B: top 15 fastest-aging cities with rising mortality
aging_rising = slopes_df[(slopes_df["age_slope"] > 0.5) & (slopes_df["mort_slope"] > 0.005)].copy()
aging_rising = aging_rising.sort_values("age_slope", ascending=False)
top_aging = aging_rising.head(15)

ax = axes[1]
bar_colors = [COLORS["danger"] if h == 0 else COLORS["primary"] for h in top_aging["has_icu"]]
y_pos = range(len(top_aging))
ax.barh(y_pos, top_aging["age_slope"], color=bar_colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([n[:20] for n in top_aging["nome"]], fontsize=8)
ax.set_xlabel("Envelhecimento (anos/ano)")
ax.set_title(f"B. Top 15 Cidades Envelhecendo Mais Rápido\n(com mortalidade subindo)", fontweight="bold")
ax.invert_yaxis()
for i, (_, r) in enumerate(top_aging.iterrows()):
    ax.text(r["age_slope"] + 0.1, i, f"mort: {r['mort_slope_pp']:+.1f}pp/yr", va="center", fontsize=7)

# Panel C: example time series for top 3 cities
ax = axes[2]
example_cities = aging_rising.nlargest(3, "n_total")
for _, ec in example_cities.iterrows():
    cdata = city_yr[(city_yr["MUNIC_MOV"] == ec["MUNIC_MOV"]) & (city_yr["n"] >= 10)].sort_values("year")
    ax.plot(cdata["year"], cdata["mean_age"], marker="o", linewidth=2, label=f"{ec['nome'][:15]} (mort: {ec['mort_slope_pp']:+.1f}pp/yr)")

ax.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax.set_xlabel("Ano")
ax.set_ylabel("Idade Média dos Pacientes J96")
ax.set_title("C. Trajetória de Envelhecimento — Top 3 por Volume", fontweight="bold")
ax.legend(fontsize=8)

fig.suptitle(f"Tendência de Envelhecimento Municipal vs Mortalidade (r = {r_trend:.2f}, p < 0.0001)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "aging_trend_vs_mortality", prefix="04")

print(f"Cidades com dados suficientes (≥5 anos, n≥10/ano): {len(slopes_df)}")
print(f"\nCorrelação trend envelhecimento vs trend mortalidade:")
print(f"  Pearson r = {r_trend:.3f}, p = {p_trend:.2e}")
print(f"  Spearman ρ = {rho_trend:.3f}, p = {p_rho_trend:.2e}")
print(f"\nCidades envelhecendo rápido (>0.5yr/yr) COM mortalidade subindo: {len(aging_rising)}")

print(f"\n{'Cidade':<25} {'Env. (yr/yr)':>12} {'Mort (pp/yr)':>13} {'Idade Média':>12} {'N':>6} {'UTI':>5}")
for _, c in aging_rising.head(15).iterrows():
    icu_str = "Sim" if c["has_icu"] else "NÃO"
    print(f"  {c['nome'][:23]:<23} {c['age_slope']:>+11.1f} {c['mort_slope_pp']:>+12.1f} {c['mean_age_overall']:>11.0f} {c['n_total']:>5} {icu_str:>5}")

# Update metrics
import json
with open("../outputs/data/metrics/04_icu_capacity.json") as f:
    m = json.load(f)
m.update({
    "trend_pearson_r": round(r_trend, 3),
    "trend_pearson_p": float(f"{p_trend:.2e}"),
    "n_cities_aging_fast_rising_mort": len(aging_rising),
    "top_aging_cities": [
        {"nome": r["nome"], "age_slope": round(r["age_slope"], 2), "mort_slope_pp": round(r["mort_slope_pp"], 1),
         "has_icu": bool(r["has_icu"]), "n": int(r["n_total"])}
        for _, r in aging_rising.head(10).iterrows()
    ],
})
save_metrics(m, "04_icu_capacity")

  Saved: 04_aging_trend_vs_mortality.png
Cidades com dados suficientes (≥5 anos, n≥10/ano): 127

Correlação trend envelhecimento vs trend mortalidade:
  Pearson r = 0.611, p = 2.26e-14
  Spearman ρ = 0.647, p = 2.02e-16

Cidades envelhecendo rápido (>0.5yr/yr) COM mortalidade subindo: 28

Cidade                    Env. (yr/yr)  Mort (pp/yr)  Idade Média      N   UTI
  Registro                       +5.4         +5.6          55   163   Sim
  Serra Negra                    +5.1         +2.6          54   169   NÃO
  Santos                         +4.6         +3.2          35  3475   Sim
  São Caetano do Sul             +4.2         +3.7          43   397   Sim
  Francisco Morato               +3.6         +4.4          35   762   NÃO
  Rio Claro                      +3.4         +2.6          29   825   NÃO
  Catanduva                      +3.2         +4.9          63   803   NÃO
  Piedade                        +2.8         +3.5          67   133   Sim
  Jaguariúna                   

## 13. Dados IBGE: Taxas Per Capita e Estrutura Etária Populacional

As análises anteriores usaram **contagens brutas** de internações J96. Isso é enganoso: uma cidade de 500 mil habitantes com 100 internações J96 tem um *problema menor* que uma cidade de 20 mil com 50 internações. Os dados do Censo IBGE 2022 e estimativas populacionais 2001–2025 permitem calcular **taxas per capita** — a métrica correta para comparação entre municípios.

Perguntas:
1. Quais cidades têm a maior **taxa de internação J96 per capita**?
2. A **proporção de idosos na população geral** prediz a mortalidade J96?
3. A lista de prioridade muda quando usamos taxas em vez de contagens brutas?

In [15]:
from shared import RAW_DATA_DIR

ibge_census = pd.read_parquet(RAW_DATA_DIR / "ibge" / "ibge_censo2022_municipios_sp.parquet")
ibge_est = pd.read_parquet(RAW_DATA_DIR / "ibge" / "ibge_estimativas_populacao_sp.parquet")

print(f"Censo 2022: {len(ibge_census)} municípios")
print(f"Estimativas: {ibge_est['ano'].nunique()} anos, {ibge_est['cod_mun_6'].nunique()} municípios")
print(f"Anos disponíveis: {sorted(ibge_est['ano'].unique())}")

# Per-capita J96 rates by municipality (total period)
city_j96 = resp.groupby("MUNIC_MOV").agg(
    n_internacoes=("MORTE", "count"),
    n_obitos=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
    mean_age_j96=("age", "mean"),
).reset_index()

city_j96 = city_j96.merge(
    ibge_census[["cod_mun_6", "nome", "pop_total", "pop_60plus", "pop_65plus",
                 "pct_60plus", "pct_65plus", "idade_media_pop", "indice_envelhecimento"]],
    left_on="MUNIC_MOV", right_on="cod_mun_6", how="inner"
)

n_years = resp["year"].nunique()
city_j96["taxa_internacao_100k"] = (city_j96["n_internacoes"] / n_years) / city_j96["pop_total"] * 100_000
city_j96["taxa_obito_100k"] = (city_j96["n_obitos"] / n_years) / city_j96["pop_total"] * 100_000

city_sig = city_j96[city_j96["n_internacoes"] >= 30].copy()

print(f"\nMunicípios com ≥30 internações J96: {len(city_sig)}")
print(f"\n--- Top 15: Taxa de Internação J96 per capita (por 100k/ano) ---")
print(f"{'Cidade':<25} {'Pop.':<10} {'J96/ano':>8} {'Taxa':>8} {'Mort%':>6} {'%60+':>6}")
for _, r in city_sig.nlargest(15, "taxa_internacao_100k").iterrows():
    print(f"  {r['nome'][:23]:<23} {r['pop_total']:>8,.0f} {r['n_internacoes']/n_years:>7.0f} "
          f"{r['taxa_internacao_100k']:>7.1f} {r['mortality']*100:>5.1f} {r['pct_60plus']:>5.1f}")

print(f"\n--- Top 15: Taxa de Óbito J96 per capita (por 100k/ano) ---")
print(f"{'Cidade':<25} {'Pop.':<10} {'Óbitos/ano':>10} {'Taxa':>8} {'%65+':>6} {'Env.':>6}")
for _, r in city_sig.nlargest(15, "taxa_obito_100k").iterrows():
    print(f"  {r['nome'][:23]:<23} {r['pop_total']:>8,.0f} {r['n_obitos']/n_years:>9.0f} "
          f"{r['taxa_obito_100k']:>7.1f} {r['pct_65plus']:>5.1f} {r['indice_envelhecimento']:>5.1f}")

Censo 2022: 645 municípios
Estimativas: 21 anos, 645 municípios
Anos disponíveis: [np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2008), np.int64(2009), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2024), np.int64(2025)]

Municípios com ≥30 internações J96: 204

--- Top 15: Taxa de Internação J96 per capita (por 100k/ano) ---
Cidade                    Pop.        J96/ano     Taxa  Mort%   %60+
  Botucatu                 145,155     421   290.3   9.1  18.2
  Paulínia                 110,537     220   199.0   4.8  14.2
  Cajuru                    23,830      31   130.1  12.9  17.8
  Bernardino de Campos      11,607      14   121.5  24.1  19.9
  Campinas                1,139,047    1094    96.0  11.4  18.4
  Ipaussu                   13,712      12    90.4  48.4  18.2
  Barra Bonita            

## 14. Idade da População Geral vs Mortalidade J96

A idade média dos *pacientes J96* correlaciona fortemente com a mortalidade (seção 9). Mas será que a **estrutura etária da população geral** (Censo IBGE) também prediz? Isso testaria se o problema é demográfico (cidades velhas produzem mais mortes J96) ou clínico (cidades que *recebem* pacientes mais velhos têm mais mortes).

In [16]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# A: Census population age vs J96 mortality
ax = axes[0, 0]
s = ax.scatter(city_sig["pct_65plus"], city_sig["mortality"] * 100,
               s=city_sig["n_internacoes"] / 5, alpha=0.5, c=COLORS["primary"], edgecolors="white", linewidth=0.3)
r_pop65, p_pop65 = stats.pearsonr(city_sig["pct_65plus"], city_sig["mortality"])
coefs = np.polyfit(city_sig["pct_65plus"], city_sig["mortality"] * 100, 1, w=np.sqrt(city_sig["n_internacoes"]))
x_line = np.linspace(city_sig["pct_65plus"].min(), city_sig["pct_65plus"].max(), 100)
ax.plot(x_line, np.polyval(coefs, x_line), "--", color="black", alpha=0.5)
ax.set_xlabel("% População ≥65 anos (Censo 2022)")
ax.set_ylabel("Mortalidade J96 (%)")
ax.set_title(f"A. População Idosa (censo) × Mortalidade J96\nr = {r_pop65:.3f}, p = {p_pop65:.2e}", fontweight="bold")

# B: Census population age vs J96 patient age (who goes to which city?)
ax = axes[0, 1]
ax.scatter(city_sig["idade_media_pop"], city_sig["mean_age_j96"],
           s=city_sig["n_internacoes"] / 5, alpha=0.5, c=COLORS["secondary"], edgecolors="white", linewidth=0.3)
r_age_pop_j96, p_age_pop_j96 = stats.pearsonr(city_sig["idade_media_pop"], city_sig["mean_age_j96"])
ax.plot([25, 50], [25, 50], ":", color="gray", alpha=0.5, label="Linha identidade")
coefs2 = np.polyfit(city_sig["idade_media_pop"], city_sig["mean_age_j96"], 1, w=np.sqrt(city_sig["n_internacoes"]))
ax.plot(x_line2 := np.linspace(30, 48, 100), np.polyval(coefs2, x_line2), "--", color="black", alpha=0.5)
ax.set_xlabel("Idade Média da População (Censo 2022)")
ax.set_ylabel("Idade Média dos Pacientes J96")
ax.set_title(f"B. Idade Pop. Geral × Idade Pacientes J96\nr = {r_age_pop_j96:.3f}, p = {p_age_pop_j96:.2e}", fontweight="bold")
ax.legend(fontsize=8)

# C: Per capita J96 rate vs % elderly
ax = axes[1, 0]
ax.scatter(city_sig["pct_60plus"], city_sig["taxa_internacao_100k"],
           s=city_sig["pop_total"] / 5000, alpha=0.5, c=COLORS["success"], edgecolors="white", linewidth=0.3)
r_rate_eld, p_rate_eld = stats.pearsonr(city_sig["pct_60plus"], city_sig["taxa_internacao_100k"])
ax.set_xlabel("% População ≥60 anos (Censo 2022)")
ax.set_ylabel("Taxa J96 (internações por 100k/ano)")
ax.set_title(f"C. Proporção de Idosos × Taxa J96 per capita\nr = {r_rate_eld:.3f}, p = {p_rate_eld:.2e}", fontweight="bold")

# D: Per capita J96 death rate vs aging index
ax = axes[1, 1]
ax.scatter(city_sig["indice_envelhecimento"], city_sig["taxa_obito_100k"],
           s=city_sig["pop_total"] / 5000, alpha=0.5, c=COLORS["danger"], edgecolors="white", linewidth=0.3)
r_death_env, p_death_env = stats.pearsonr(city_sig["indice_envelhecimento"], city_sig["taxa_obito_100k"])
ax.set_xlabel("Índice de Envelhecimento (65+ / 0-14 × 100)")
ax.set_ylabel("Taxa de Óbitos J96 (por 100k/ano)")
ax.set_title(f"D. Índice de Envelhecimento × Taxa de Óbitos J96\nr = {r_death_env:.3f}, p = {p_death_env:.2e}", fontweight="bold")

fig.suptitle("Dados IBGE Censo 2022 × Mortalidade por Insuficiência Respiratória (J96)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "ibge_population_vs_j96", prefix="04")

print(f"Correlações (pop. geral × J96):")
print(f"  %65+ × mortalidade J96:     r = {r_pop65:.3f}, p = {p_pop65:.2e}")
print(f"  Idade pop × idade J96:      r = {r_age_pop_j96:.3f}, p = {p_age_pop_j96:.2e}")
print(f"  %60+ × taxa J96/100k:       r = {r_rate_eld:.3f}, p = {p_rate_eld:.2e}")
print(f"  Índ. envelhecimento × óbito J96/100k: r = {r_death_env:.3f}, p = {p_death_env:.2e}")

  Saved: 04_ibge_population_vs_j96.png
Correlações (pop. geral × J96):
  %65+ × mortalidade J96:     r = 0.051, p = 4.69e-01
  Idade pop × idade J96:      r = 0.207, p = 2.97e-03
  %60+ × taxa J96/100k:       r = 0.196, p = 4.91e-03
  Índ. envelhecimento × óbito J96/100k: r = 0.317, p = 3.81e-06


## 15. R² com IBGE: Idade Populacional vs Idade Clínica vs UTI

A seção 11 mostrou que a idade *dos pacientes J96* explica 97% da variância. Agora testamos: a idade *da população geral* (Censo IBGE) tem o mesmo poder preditivo? Isso distingue dois cenários:

- **Cenário A:** A estrutura demográfica municipal *determina* a mortalidade J96 → política = geriatria + prevenção
- **Cenário B:** A idade clínica importa mas a demográfica não → problema é referência e captação, não demografia

In [17]:
# R² decomposition: clinical age vs census age vs ICU
cs = city_sig.dropna(subset=["mean_age_j96", "pct_65plus"]).copy()
cs["has_icu"] = cs["MUNIC_MOV"].isin(
    resp[resp["icu_beds_sus"] > 0]["MUNIC_MOV"].unique()
).astype(int)

y = cs["mortality"].values
w = cs["n_internacoes"].values.astype(float)

def weighted_r2(X, y, w):
    W = np.diag(np.sqrt(w))
    Xw = W @ X
    yw = W @ y
    beta, _, _, _ = np.linalg.lstsq(Xw, yw, rcond=None)
    y_pred = X @ beta
    ss_res = np.sum(w * (y - y_pred) ** 2)
    ss_tot = np.sum(w * (y - np.average(y, weights=w)) ** 2)
    return 1 - ss_res / ss_tot

# Individual models
X_clinical = np.column_stack([cs["mean_age_j96"].values, np.ones(len(cs))])
X_census = np.column_stack([cs["pct_65plus"].values, np.ones(len(cs))])
X_icu = np.column_stack([cs["has_icu"].values, np.ones(len(cs))])
X_full = np.column_stack([cs["mean_age_j96"].values, cs["pct_65plus"].values, cs["has_icu"].values, np.ones(len(cs))])
X_clinical_census = np.column_stack([cs["mean_age_j96"].values, cs["pct_65plus"].values, np.ones(len(cs))])

r2_clinical = weighted_r2(X_clinical, y, w)
r2_census = weighted_r2(X_census, y, w)
r2_icu = weighted_r2(X_icu, y, w)
r2_full = weighted_r2(X_full, y, w)
r2_clin_census = weighted_r2(X_clinical_census, y, w)

print("R² Decomposition — Municipal J96 Mortality")
print(f"{'Modelo':<40} {'R²':>8} {'% do completo':>14}")
print("-" * 65)
models = [
    ("Idade clínica J96 (mean_age_j96)", r2_clinical),
    ("Idade populacional (%65+ Censo)", r2_census),
    ("UTI (presença)", r2_icu),
    ("Idade clínica + Idade censo", r2_clin_census),
    ("Completo (clínica + censo + UTI)", r2_full),
]
for name, r2 in models:
    pct = r2 / r2_full * 100 if r2_full > 0 else 0
    print(f"  {name:<38} {r2:>7.3f} {pct:>13.0f}%")

print(f"\n→ Idade clínica (dos pacientes J96) é o preditor dominante.")
print(f"  A idade da população geral (Censo) contribui R² = {r2_census:.3f},")
print(f"  mas a idade clínica sozinha já captura R² = {r2_clinical:.3f}.")
print(f"  Adicionar dados do Censo à idade clínica {'melhora' if r2_clin_census > r2_clinical + 0.01 else 'pouco acrescenta ao'} o modelo.")

R² Decomposition — Municipal J96 Mortality
Modelo                                         R²  % do completo
-----------------------------------------------------------------
  Idade clínica J96 (mean_age_j96)         0.633            96%
  Idade populacional (%65+ Censo)          0.016             2%
  UTI (presença)                           0.042             6%
  Idade clínica + Idade censo              0.645            97%
  Completo (clínica + censo + UTI)         0.663           100%

→ Idade clínica (dos pacientes J96) é o preditor dominante.
  A idade da população geral (Censo) contribui R² = 0.016,
  mas a idade clínica sozinha já captura R² = 0.633.
  Adicionar dados do Censo à idade clínica melhora o modelo.


## 16. Lista de Prioridade Revisada (Per Capita com IBGE)

A lista original (seção 10) usava contagens brutas. Agora ranqueamos pela **taxa de óbitos J96 por 100k habitantes/ano** — a métrica comparável entre municípios de tamanhos diferentes.

In [18]:
# Priority list with per-capita rates
cs["has_icu_label"] = cs["has_icu"].map({1: "Sim", 0: "NÃO"})
cs["score_percapita"] = cs["taxa_obito_100k"] * (1 + (1 - cs["has_icu"]) * 0.5)

top30 = cs.nlargest(30, "score_percapita")

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Left: bar chart top 30
ax = axes[0]
bar_colors = [COLORS["danger"] if h == 0 else COLORS["primary"] for h in top30["has_icu"]]
y_pos = range(len(top30))
bars = ax.barh(y_pos, top30["taxa_obito_100k"], color=bar_colors, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{n[:20]} ({p/1000:.0f}k)" for n, p in zip(top30["nome"], top30["pop_total"])], fontsize=8)
ax.set_xlabel("Óbitos J96 por 100k hab./ano")
ax.set_title("A. Top 30 Municípios: Taxa de Óbito J96 per Capita", fontweight="bold")
ax.invert_yaxis()
for i, (_, r) in enumerate(top30.iterrows()):
    ax.text(r["taxa_obito_100k"] + 0.3, i, f"UTI: {r['has_icu_label']} | {r['pct_65plus']:.0f}%≥65",
            va="center", fontsize=7)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=COLORS["danger"], label="Sem UTI"),
                   Patch(facecolor=COLORS["primary"], label="Com UTI")],
          loc="lower right", fontsize=9)

# Right: scatter — rate vs population with color=has_icu
ax = axes[1]
no_icu = cs[cs["has_icu"] == 0]
with_icu = cs[cs["has_icu"] == 1]
ax.scatter(no_icu["pop_total"], no_icu["taxa_obito_100k"],
           s=no_icu["pct_65plus"] * 3, alpha=0.5, c=COLORS["danger"],
           edgecolors="white", linewidth=0.3, label=f"Sem UTI ({len(no_icu)})")
ax.scatter(with_icu["pop_total"], with_icu["taxa_obito_100k"],
           s=with_icu["pct_65plus"] * 3, alpha=0.5, c=COLORS["primary"],
           edgecolors="white", linewidth=0.3, label=f"Com UTI ({len(with_icu)})")
ax.set_xscale("log")
ax.set_xlabel("População (log)")
ax.set_ylabel("Óbitos J96 por 100k hab./ano")
ax.set_title("B. Taxa per Capita × Tamanho do Município", fontweight="bold")
ax.legend(fontsize=9)
for _, r in top30.head(10).iterrows():
    ax.annotate(r["nome"][:15], (r["pop_total"], r["taxa_obito_100k"]),
                fontsize=7, textcoords="offset points", xytext=(5, 3))

fig.suptitle("Prioridade Revisada: Óbitos J96 per Capita (IBGE Censo 2022)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "ibge_priority_percapita", prefix="04")

print(f"\n--- Top 20 Municípios: Maior Taxa de Óbito J96 per capita ---")
print(f"{'#':>3} {'Cidade':<25} {'Pop.':<10} {'J96/ano':>8} {'Óbitos/ano':>10} {'Óbito/100k':>10} {'%65+':>6} {'UTI':>5}")
for rank, (_, r) in enumerate(top30.head(20).iterrows(), 1):
    print(f"{rank:>3}. {r['nome'][:23]:<23} {r['pop_total']:>8,.0f} "
          f"{r['n_internacoes']/n_years:>7.0f} {r['n_obitos']/n_years:>9.1f} "
          f"{r['taxa_obito_100k']:>9.1f} {r['pct_65plus']:>5.1f} {r['has_icu_label']:>5}")

  Saved: 04_ibge_priority_percapita.png

--- Top 20 Municípios: Maior Taxa de Óbito J96 per capita ---
  # Cidade                    Pop.        J96/ano Óbitos/ano Óbito/100k   %65+   UTI
  1. Barra Bonita              34,346      30      22.0      64.1  14.9   NÃO
  2. Catanduva                115,791      80      66.2      57.2  14.4   NÃO
  3. Ipaussu                   13,712      12       6.0      43.8  12.5   NÃO
  4. Rancharia                 28,588      16      11.3      39.5  13.5   NÃO
  5. Tupi Paulista             15,854       7       5.2      32.8  16.9   NÃO
  6. Pariquera-Açu             19,233      17       9.4      48.9  12.8   Sim
  7. Santa Isabel              53,174      39      17.0      32.0  11.3   NÃO
  8. Aparecida                 32,569      20      10.1      31.0  13.3   NÃO
  9. São José do Rio Pardo     52,205      29      15.8      30.3  15.3   NÃO
 10. Bernardino de Campos      11,607      14       3.4      29.3  14.7   NÃO
 11. Águas de Lindóia          1

## 17. Tendência Temporal: População Crescendo vs Internações J96 Crescendo

Usando as estimativas IBGE (2001–2025) podemos verificar: as internações J96 estão crescendo *mais rápido* que a população? Se sim, a crise é real e não apenas um reflexo de crescimento demográfico.

In [19]:
# J96 admissions per year vs population estimates
j96_annual = resp.groupby(resp["year"].astype(int)).agg(
    n_internacoes=("MORTE", "count"),
    n_obitos=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
).reset_index()

pop_annual = ibge_est.groupby("ano")["pop_estimada"].sum().reset_index()
pop_annual.columns = ["year", "pop_sp"]

# Merge — only years present in both
j96_pop = j96_annual.merge(pop_annual, on="year", how="inner")
j96_pop["taxa_internacao_100k"] = j96_pop["n_internacoes"] / j96_pop["pop_sp"] * 100_000
j96_pop["taxa_obito_100k"] = j96_pop["n_obitos"] / j96_pop["pop_sp"] * 100_000

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# A: Absolute numbers (admissions + deaths)
ax = axes[0]
ax.bar(j96_pop["year"], j96_pop["n_internacoes"], color=COLORS["primary"], alpha=0.7, label="Internações")
ax.bar(j96_pop["year"], j96_pop["n_obitos"], color=COLORS["danger"], alpha=0.8, label="Óbitos")
ax.set_xlabel("Ano")
ax.set_ylabel("N")
ax.set_title("A. Volume Absoluto J96", fontweight="bold")
ax.legend()

# B: Per-capita rates
ax = axes[1]
ax.plot(j96_pop["year"], j96_pop["taxa_internacao_100k"], "o-",
        color=COLORS["primary"], linewidth=2, label="Internações/100k")
ax2 = ax.twinx()
ax2.plot(j96_pop["year"], j96_pop["taxa_obito_100k"], "s-",
         color=COLORS["danger"], linewidth=2, label="Óbitos/100k")
ax.set_xlabel("Ano")
ax.set_ylabel("Internações J96 por 100k", color=COLORS["primary"])
ax2.set_ylabel("Óbitos J96 por 100k", color=COLORS["danger"])
ax.set_title("B. Taxas Per Capita (IBGE)", fontweight="bold")
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

# C: Mortality rate over time
ax = axes[2]
ax.plot(j96_pop["year"], j96_pop["mortality"] * 100, "D-",
        color=COLORS["danger"], linewidth=2.5, markersize=8)
ax.axhline(j96_pop[j96_pop["year"] <= 2019]["mortality"].mean() * 100,
           color="gray", linestyle="--", alpha=0.5, label="Média pré-COVID")
ax.fill_between(j96_pop["year"], j96_pop["mortality"] * 100,
                j96_pop[j96_pop["year"] <= 2019]["mortality"].mean() * 100,
                where=j96_pop["mortality"] * 100 > j96_pop[j96_pop["year"] <= 2019]["mortality"].mean() * 100,
                alpha=0.15, color=COLORS["danger"])
ax.set_xlabel("Ano")
ax.set_ylabel("Mortalidade (%)")
ax.set_title("C. Mortalidade J96 (% dos internados)", fontweight="bold")
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax.legend(fontsize=8)

fig.suptitle("Insuficiência Respiratória (J96) — Tendência Temporal com Denominador IBGE",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "ibge_temporal_trend", prefix="04")

# Compute growth rates
pre_covid = j96_pop[j96_pop["year"].between(2016, 2019)]
post_covid = j96_pop[j96_pop["year"].between(2022, 2025)]

if len(pre_covid) > 0 and len(post_covid) > 0:
    taxa_pre = pre_covid["taxa_internacao_100k"].mean()
    taxa_post = post_covid["taxa_internacao_100k"].mean()
    obito_pre = pre_covid["taxa_obito_100k"].mean()
    obito_post = post_covid["taxa_obito_100k"].mean()
    print(f"\nTaxa de internação J96 per capita:")
    print(f"  Pré-COVID (2016-2019): {taxa_pre:.1f} por 100k/ano")
    print(f"  Pós-COVID (2022-2025): {taxa_post:.1f} por 100k/ano")
    print(f"  Variação: {(taxa_post/taxa_pre - 1)*100:+.1f}%")
    print(f"\nTaxa de óbito J96 per capita:")
    print(f"  Pré-COVID: {obito_pre:.1f} por 100k/ano")
    print(f"  Pós-COVID: {obito_post:.1f} por 100k/ano")
    print(f"  Variação: {(obito_post/obito_pre - 1)*100:+.1f}%")

print(f"\n--- Tabela completa ---")
print(j96_pop[["year", "pop_sp", "n_internacoes", "n_obitos", "taxa_internacao_100k", "taxa_obito_100k", "mortality"]].to_string(index=False))

  Saved: 04_ibge_temporal_trend.png

Taxa de internação J96 per capita:
  Pré-COVID (2016-2019): 23.7 por 100k/ano
  Pós-COVID (2022-2025): 24.9 por 100k/ano
  Variação: +5.1%

Taxa de óbito J96 per capita:
  Pré-COVID: 7.3 por 100k/ano
  Pós-COVID: 9.1 por 100k/ano
  Variação: +23.5%

--- Tabela completa ---
 year   pop_sp  n_internacoes  n_obitos  taxa_internacao_100k  taxa_obito_100k  mortality
 2016 44749699          10891      3428             24.337594         7.660387   0.314755
 2017 45094866          10764      3338             23.869680         7.402173   0.310108
 2018 45538936          10772      3236             23.654483         7.106007   0.300408
 2019 45919049          10466      3290             22.792284         7.164783   0.314351
 2020 46289333          14681      4944             31.715730        10.680646   0.336762
 2021 46649132          13195      4135             28.285628         8.864045   0.313376
 2024 45973194          11612      4321             25.2581

In [20]:
# Update metrics with IBGE findings
import json
with open("../outputs/data/metrics/04_icu_capacity.json") as f:
    m = json.load(f)

m.update({
    "ibge_r_pop65_mortality": round(r_pop65, 3),
    "ibge_r_pop65_mortality_p": float(f"{p_pop65:.2e}"),
    "ibge_r_age_pop_j96": round(r_age_pop_j96, 3),
    "ibge_r_rate_elderly": round(r_rate_eld, 3),
    "ibge_r2_clinical_age": round(r2_clinical, 3),
    "ibge_r2_census_age": round(r2_census, 3),
    "ibge_r2_full_model": round(r2_full, 3),
    "ibge_taxa_pre_covid_100k": round(taxa_pre, 1) if 'taxa_pre' in dir() else None,
    "ibge_taxa_post_covid_100k": round(taxa_post, 1) if 'taxa_post' in dir() else None,
    "ibge_top_percapita_cities": [
        {"nome": r["nome"], "pop_total": int(r["pop_total"]),
         "taxa_obito_100k": round(r["taxa_obito_100k"], 1),
         "pct_65plus": round(r["pct_65plus"], 1),
         "has_icu": bool(r["has_icu"])}
        for _, r in top30.head(10).iterrows()
    ],
})
save_metrics(m, "04_icu_capacity")
print("Metrics updated with IBGE data.")

  Saved metrics: 04_icu_capacity.json
Metrics updated with IBGE data.
